# 01 Image Classifier Baseline

CNN classifier template based on the MNIST/CIFAR bootcamp notebooks.

In [ ]:
# Colab setup if needed
# from google.colab import drive
# drive.mount("/content/drive")
# !pip install -q -e .

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

from cnugs_ml.data import prepare_images, describe_array, set_global_seed
from cnugs_ml.models import build_cnn_classifier
from cnugs_ml.plots import plot_image_grid, plot_loss_history, plot_confusion_matrix
from cnugs_ml.train import make_early_stopping, predict_classes
from cnugs_ml.submission import save_classification_submission

set_global_seed(42)

## Load and prepare data

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

x_train = prepare_images(train_images, add_channel=True)
x_test = prepare_images(test_images, add_channel=True)

describe_array("x_train", x_train)
describe_array("train_labels", train_labels)
plot_image_grid(x_train, train_labels, n=6, rows=2)

## Build baseline CNN

In [ ]:
model = build_cnn_classifier(
    input_shape=x_train.shape[1:],
    num_classes=10,
    conv_filters=(32, 64, 64),
    dense_units=(128,),
    learning_rate=1e-3,
)
model.summary()

## Train

In [ ]:
early_stopping = make_early_stopping(monitor="val_loss", patience=5)
history = model.fit(
    x_train, train_labels,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping],
)

In [ ]:
plot_loss_history(history)

## Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, test_labels)
print(f"test_loss = {test_loss:.5f}")
print(f"test_acc  = {test_acc:.5f}")

y_pred = predict_classes(model, x_test)
plot_confusion_matrix(test_labels, y_pred, class_names=[str(i) for i in range(10)])

## Save predictions

In [ ]:
submission = save_classification_submission(y_pred, "../outputs/classifier_submission.csv")
submission.head()